# 03 — Define Summarize Composite / 03 Define Summarize Composite

**Chapter 25 — File 3 of 3 / 第25章 — 第3个文件（共3个）**

---

## Summary / 总结

This script demonstrates **example of defining composite models for training cyclegan generators**.

本脚本演示 **example of defining composite models for training cyclegan generators**。

---
## Step 1 — example of defining composite models for training cyclegan generators

In [ ]:
from keras.optimizers import Adam
from keras.models import Model
from keras.models import Input
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import Activation
from keras.layers import LeakyReLU
from keras.initializers import RandomNormal
from keras.layers import Concatenate
from keras_contrib.layers.normalization.instancenormalization import InstanceNormalization

---
## Step 2 — define the discriminator model

In [ ]:
def define_discriminator(image_shape):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — source image input

In [ ]:
in_image = Input(shape=image_shape)

---
## Step 5 — C64

In [ ]:
d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 6 — C128

In [ ]:
d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 7 — C256

In [ ]:
d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 8 — C512

In [ ]:
d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 9 — second last output layer

In [ ]:
d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 10 — patch output

In [ ]:
patch_out = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)

---
## Step 11 — define model

In [ ]:
model = Model(in_image, patch_out)

---
## Step 12 — compile model

In [ ]:
model.compile(loss='mse', optimizer=Adam(lr=0.0002, beta_1=0.5), loss_weights=[0.5])
	return model

---
## Step 13 — generator a resnet block

In [ ]:
def resnet_block(n_filters, input_layer):

---
## Step 14 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 15 — first layer convolutional layer

In [ ]:
g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(input_layer)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 16 — second convolutional layer

In [ ]:
g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)

---
## Step 17 — concatenate merge channel-wise with input layer

In [ ]:
g = Concatenate()([g, input_layer])
	return g

---
## Step 18 — define the standalone generator model

In [ ]:
def define_generator(image_shape, n_resnet=9):

---
## Step 19 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 20 — image input

In [ ]:
in_image = Input(shape=image_shape)

---
## Step 21 — c7s1-64

In [ ]:
g = Conv2D(64, (7,7), padding='same', kernel_initializer=init)(in_image)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 22 — d128

In [ ]:
g = Conv2D(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 23 — d256

In [ ]:
g = Conv2D(256, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 24 — R256

In [ ]:
for _ in range(n_resnet):
		g = resnet_block(256, g)

---
## Step 25 — u128

In [ ]:
g = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 26 — u64

In [ ]:
g = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)

---
## Step 27 — c7s1-3

In [ ]:
g = Conv2D(3, (7,7), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	out_image = Activation('tanh')(g)

---
## Step 28 — define model

In [ ]:
model = Model(in_image, out_image)
	return model

---
## Step 29 — define a composite model for updating generators by adversarial and cycle loss

In [ ]:
def define_composite_model(g_model_1, d_model, g_model_2, image_shape):

---
## Step 30 — ensure the model we're updating is trainable

In [ ]:
g_model_1.trainable = True

---
## Step 31 — mark discriminator as not trainable

In [ ]:
d_model.trainable = False

---
## Step 32 — mark other generator model as not trainable

In [ ]:
g_model_2.trainable = False

---
## Step 33 — discriminator element

In [ ]:
input_gen = Input(shape=image_shape)
	gen1_out = g_model_1(input_gen)
	output_d = d_model(gen1_out)

---
## Step 34 — identity element

In [ ]:
input_id = Input(shape=image_shape)
	output_id = g_model_1(input_id)

---
## Step 35 — forward cycle

In [ ]:
output_f = g_model_2(gen1_out)

---
## Step 36 — backward cycle

In [ ]:
gen2_out = g_model_2(input_id)
	output_b = g_model_1(gen2_out)

---
## Step 37 — define model graph

In [ ]:
model = Model([input_gen, input_id], [output_d, output_id, output_f, output_b])

---
## Step 38 — define optimization algorithm configuration

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)

---
## Step 39 — compile model with weighting of least squares loss and L1 loss

In [ ]:
model.compile(loss=['mse', 'mae', 'mae', 'mae'], loss_weights=[1, 5, 10, 10], optimizer=opt)
	return model

---
## Step 40 — input shape

In [ ]:
image_shape = (256,256,3)

---
## Step 41 — generator: A -> B

In [ ]:
g_model_AtoB = define_generator(image_shape)

---
## Step 42 — generator: B -> A

In [ ]:
g_model_BtoA = define_generator(image_shape)

---
## Step 43 — discriminator: A -> [real/fake]

In [ ]:
d_model_A = define_discriminator(image_shape)

---
## Step 44 — discriminator: B -> [real/fake]

In [ ]:
d_model_B = define_discriminator(image_shape)

---
## Step 45 — composite: A -> B -> [real/fake, A]

In [ ]:
c_model_AtoB = define_composite_model(g_model_AtoB, d_model_B, g_model_BtoA, image_shape)

---
## Step 46 — composite: B -> A -> [real/fake, B]

In [ ]:
c_model_BtoA = define_composite_model(g_model_BtoA, d_model_A, g_model_AtoB, image_shape)

---
## Learning Notes / 学习笔记

- **概念**: example of defining composite models for training cyclegan generators 是机器学习中的常用技术。  
  *example of defining composite models for training cyclegan generators is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Define Summarize Composite / 03 Define Summarize Composite
# Complete Code / 完整代码
# ===============================

# example of defining composite models for training cyclegan generators
from keras.optimizers import Adam
from keras.models import Model
from keras.models import Input
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import Activation
from keras.layers import LeakyReLU
from keras.initializers import RandomNormal
from keras.layers import Concatenate
from keras_contrib.layers.normalization.instancenormalization import InstanceNormalization

# define the discriminator model
def define_discriminator(image_shape):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# source image input
	in_image = Input(shape=image_shape)
	# C64
	d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	d = LeakyReLU(alpha=0.2)(d)
	# C128
	d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C256
	d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C512
	d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# second last output layer
	d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = InstanceNormalization(axis=-1)(d)
	d = LeakyReLU(alpha=0.2)(d)
	# patch output
	patch_out = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)
	# define model
	model = Model(in_image, patch_out)
	# compile model
	model.compile(loss='mse', optimizer=Adam(lr=0.0002, beta_1=0.5), loss_weights=[0.5])
	return model

# generator a resnet block
def resnet_block(n_filters, input_layer):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# first layer convolutional layer
	g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(input_layer)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# second convolutional layer
	g = Conv2D(n_filters, (3,3), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	# concatenate merge channel-wise with input layer
	g = Concatenate()([g, input_layer])
	return g

# define the standalone generator model
def define_generator(image_shape, n_resnet=9):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image input
	in_image = Input(shape=image_shape)
	# c7s1-64
	g = Conv2D(64, (7,7), padding='same', kernel_initializer=init)(in_image)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# d128
	g = Conv2D(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# d256
	g = Conv2D(256, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# R256
	for _ in range(n_resnet):
		g = resnet_block(256, g)
	# u128
	g = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# u64
	g = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	g = Activation('relu')(g)
	# c7s1-3
	g = Conv2D(3, (7,7), padding='same', kernel_initializer=init)(g)
	g = InstanceNormalization(axis=-1)(g)
	out_image = Activation('tanh')(g)
	# define model
	model = Model(in_image, out_image)
	return model

# define a composite model for updating generators by adversarial and cycle loss
def define_composite_model(g_model_1, d_model, g_model_2, image_shape):
	# ensure the model we're updating is trainable
	g_model_1.trainable = True
	# mark discriminator as not trainable
	d_model.trainable = False
	# mark other generator model as not trainable
	g_model_2.trainable = False
	# discriminator element
	input_gen = Input(shape=image_shape)
	gen1_out = g_model_1(input_gen)
	output_d = d_model(gen1_out)
	# identity element
	input_id = Input(shape=image_shape)
	output_id = g_model_1(input_id)
	# forward cycle
	output_f = g_model_2(gen1_out)
	# backward cycle
	gen2_out = g_model_2(input_id)
	output_b = g_model_1(gen2_out)
	# define model graph
	model = Model([input_gen, input_id], [output_d, output_id, output_f, output_b])
	# define optimization algorithm configuration
	opt = Adam(lr=0.0002, beta_1=0.5)
	# compile model with weighting of least squares loss and L1 loss
	model.compile(loss=['mse', 'mae', 'mae', 'mae'], loss_weights=[1, 5, 10, 10], optimizer=opt)
	return model

# input shape
image_shape = (256,256,3)
# generator: A -> B
g_model_AtoB = define_generator(image_shape)
# generator: B -> A
g_model_BtoA = define_generator(image_shape)
# discriminator: A -> [real/fake]
d_model_A = define_discriminator(image_shape)
# discriminator: B -> [real/fake]
d_model_B = define_discriminator(image_shape)
# composite: A -> B -> [real/fake, A]
c_model_AtoB = define_composite_model(g_model_AtoB, d_model_B, g_model_BtoA, image_shape)
# composite: B -> A -> [real/fake, B]
c_model_BtoA = define_composite_model(g_model_BtoA, d_model_A, g_model_AtoB, image_shape)